In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectFromModel

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LassoCV, LogisticRegression
from sklearn.ensemble import (
    RandomForestRegressor, RandomForestClassifier,
    BaggingRegressor, AdaBoostRegressor, GradientBoostingRegressor,
)
from sklearn.svm import SVC, LinearSVC
from xgboost import XGBRegressor
from lightgbm import LGBMClassifier

NOTEBOOK_DIR = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Notebook/20261021_Github2"
RANDOM_STATE = 42

features = pd.read_csv(f"{NOTEBOOK_DIR}/step1_features.csv")
targets = pd.read_csv(f"{NOTEBOOK_DIR}/step1_targets.csv")
data = features.merge(targets, on="T_ID")
assert len(data) == 10000
data.shape

## Preprocessing + feature selection pipeline

One consistent pipeline is used regardless of downstream estimator (simpler than a per-model-family variant): median-impute + scale numeric columns, most-frequent-impute + one-hot encode categorical columns, then an embedded L1-based feature selector (`LassoCV` for regression targets, L1-`LogisticRegression` for classification targets). All steps are fit only on the training fold via the `Pipeline`/`ColumnTransformer`, so no leakage from validation/test data.

In [ ]:
NUMERIC_COLS = [
    "T_AGE", "T_SBP", "T_DBP", "T_PULSE", "T_BMI", "T_GLU0",
    "T_TCHL", "T_HDL", "T_LDL", "T_TG",
    "HTN_DUR", "DM_DUR", "LIP_DUR", "T_SMDUYR", "T_SMAM", "T_DRDU",
    # cardiometabolic PRS (out-of-fold, not derived from any atrophy target) - same for every target
    "PRS_HTN", "PRS_DM", "PRS_LIP",
]
CATEGORICAL_COLS = [
    "T_SEX", "T_MARRY", "T_HTN", "T_DM", "T_LIP",
    "T_FMFHT1", "T_FMFHT2", "T_FMFDM1", "T_FMFDM2",
    "T_SMOKE", "T_PSM", "T_DRINK", "T_EXER", "apoe_geno_group",
]
def make_preprocessor(numeric_cols):
    return ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORICAL_COLS),
    ])


# Cache fitted preprocess/select steps on disk: during a hyperparameter search only the final
# model's params change across trials, so the (LassoCV-based) selector is fit once per fold, not per trial.
import tempfile
from joblib import Memory
CACHE_DIR = tempfile.mkdtemp(prefix="step1_pipe_cache_")
memory = Memory(CACHE_DIR, verbose=0)


def make_pipeline(estimator, task):
    # clone() is essential here: estimator templates are reused across multiple
    # targets/loop iterations, and refitting the *same* instance repeatedly trips up
    # stateful validation in some estimators (e.g. LightGBM checks feature count against
    # its previous fit and errors on mismatch since feature selection varies per target).
    from sklearn.base import clone
    selector_model = LassoCV(cv=5, random_state=RANDOM_STATE) if task == "regression" \
        else LogisticRegression(penalty="l1", solver="liblinear", random_state=RANDOM_STATE)
    return Pipeline([
        ("preprocess", make_preprocessor(NUMERIC_COLS)),
        ("select", SelectFromModel(selector_model)),
        ("model", clone(estimator)),
    ], memory=memory)


def features_for(target_col):
    return data[NUMERIC_COLS + CATEGORICAL_COLS]

## Hyperparameter search

Each (target, model) pair is tuned with `RandomizedSearchCV` (`N_ITER` random draws, 3-fold inner CV) on the **training split only**; the held-out test split is touched once, by the refit best model. Reported per row:
- `inner_cv_best` - best mean inner-CV score found by the search (slightly optimistic: it's the max over draws).
- `cv_*_mean/std` - 5-fold CV of the tuned configuration on the training split (same protocol as 20260918, for comparability; also mildly optimistic since the params were chosen on the same data).
- `test_*` - held-out test metrics - **the number to compare against 20260918**.
- `best_params` - chosen hyperparameters.

Regression and classification use separate search spaces. Models whose regularization is already CV-tuned internally (`LassoCV`, `ElasticNetCV` with an `l1_ratio` grid) or that have no hyperparameters (`LinearRegression`) are fit directly. For classification the L1-logistic feature-selector strength (`select__estimator__C`) is tuned jointly with the model.

The linear SVM now uses `LinearSVC` instead of `SVC(kernel="linear")` - same model family (linear SVM, squared-hinge loss), but its solver scales to n=8,000 x many search draws, whereas the kernel-SVC solver is too slow for tuning.

In [ ]:
from scipy.stats import loguniform, randint, uniform
import time

N_ITER = 30
INNER_CV = 3


def run_search(pipe, space, X_train, y_train, inner_cv, scoring):
    if not space:
        return pipe.fit(X_train, y_train), np.nan, {}
    search = RandomizedSearchCV(
        pipe, space, n_iter=N_ITER, cv=inner_cv, scoring=scoring,
        random_state=RANDOM_STATE, n_jobs=-1, refit=True,
    )
    search.fit(X_train, y_train)
    best_params = {k.replace("model__", "").replace("select__estimator__", "selector_"): v
                   for k, v in search.best_params_.items()}
    return search.best_estimator_, search.best_score_, best_params

## Regression track

80/20 train/test split, then 5-fold CV within training for each model, reporting mean CV R² alongside held-out test R²/RMSE/MAE.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LassoCV, ElasticNetCV

# n_jobs=1 on the estimators: parallelism comes from the search / CV level instead (avoids oversubscription)
REGRESSION_MODELS = {
    "linear_regression": LinearRegression(),
    "ridge": Ridge(random_state=RANDOM_STATE),
    # CV-tuned internally - no outer search needed
    "lasso": LassoCV(cv=5, random_state=RANDOM_STATE),
    "elastic_net": ElasticNetCV(cv=5, l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0], random_state=RANDOM_STATE),
    "random_forest": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1),
    "bagging": BaggingRegressor(random_state=RANDOM_STATE, n_jobs=1),
    "adaboost": AdaBoostRegressor(random_state=RANDOM_STATE),
    "gradient_boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "xgboost": XGBRegressor(random_state=RANDOM_STATE, n_jobs=1, verbosity=0),
}

REGRESSION_SPACES = {
    "linear_regression": {},
    "ridge": {"model__alpha": loguniform(1e-3, 1e3)},
    "lasso": {},
    "elastic_net": {},
    "random_forest": {
        "model__n_estimators": randint(200, 800),
        "model__max_depth": [None, 5, 8, 12, 20],
        "model__min_samples_leaf": randint(1, 60),
        "model__max_features": [1.0, "sqrt", 0.3, 0.5, 0.7],
    },
    "bagging": {
        "model__n_estimators": randint(20, 300),
        "model__max_samples": uniform(0.3, 0.7),
        "model__max_features": uniform(0.3, 0.7),
    },
    "adaboost": {
        "model__n_estimators": randint(50, 500),
        "model__learning_rate": loguniform(1e-2, 1.0),
        "model__loss": ["linear", "square", "exponential"],
    },
    "gradient_boosting": {
        "model__n_estimators": randint(100, 800),
        "model__learning_rate": loguniform(5e-3, 0.2),
        "model__max_depth": randint(2, 6),
        "model__subsample": uniform(0.5, 0.5),
        "model__min_samples_leaf": randint(5, 100),
    },
    "xgboost": {
        "model__n_estimators": randint(100, 1000),
        "model__learning_rate": loguniform(5e-3, 0.2),
        "model__max_depth": randint(2, 7),
        "model__subsample": uniform(0.5, 0.5),
        "model__colsample_bytree": uniform(0.5, 0.5),
        "model__min_child_weight": loguniform(1, 50),
        "model__reg_lambda": loguniform(0.1, 100),
    },
}

TARGET_NAMES = ["composite", "hippo", "wholebrain", "ventricles", "amygdala"]
REGRESSION_TARGETS = [f"target_{t}_cont" for t in TARGET_NAMES]

reg_results = []
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_kf = KFold(n_splits=INNER_CV, shuffle=True, random_state=RANDOM_STATE)

for target_col in REGRESSION_TARGETS:
    X, y = features_for(target_col), data[target_col]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

    for name, estimator in REGRESSION_MODELS.items():
        t0 = time.time()
        pipe = make_pipeline(estimator, task="regression")
        best_pipe, inner_best, best_params = run_search(
            pipe, REGRESSION_SPACES[name], X_train, y_train, inner_kf, "r2"
        )
        cv_scores = cross_val_score(best_pipe, X_train, y_train, cv=kf, scoring="r2", n_jobs=-1)
        y_pred = best_pipe.predict(X_test)

        reg_results.append({
            "target": target_col,
            "model": name,
            "inner_cv_best": inner_best,
            "cv_r2_mean": cv_scores.mean(),
            "cv_r2_std": cv_scores.std(),
            "test_r2": r2_score(y_test, y_pred),
            "test_rmse": mean_squared_error(y_test, y_pred, squared=False),
            "test_mae": mean_absolute_error(y_test, y_pred),
            "best_params": str(best_params),
        })
        print(f"{target_col:42s} {name:20s} cv_r2={cv_scores.mean():.3f}  test_r2={reg_results[-1]['test_r2']:.3f}"
              f"  ({time.time() - t0:.0f}s)", flush=True)

reg_results_df = pd.DataFrame(reg_results)

In [ ]:
reg_results_df.sort_values(["target", "test_r2"], ascending=[True, False])

## Classification track (binary: top-X% vs rest)

Same train/test split logic, stratified by label. Runs logistic regression (cheap linear reference), SVM (linear via `LinearSVC` + RBF), random forest, and LightGBM on 15 labels (5 targets x top 20% / 10% / 5% vs the rest; positive rates 0.20 / 0.10 / 0.05). Hyperparameters are tuned on ROC-AUC.

**Validation** (5-fold stratified CV of the tuned configuration on the training split): ROC-AUC and AUPRC (average precision), mean and SD over folds. Compare AUPRC with `pos_rate`, the AUPRC of a random classifier.

**Decision threshold**: the default 0.5 cut-off predicts almost no positives when only 5-10% are positive, so the threshold is chosen on the training split - the cut-off maximizing Youden's J (sensitivity + specificity - 1) on the pooled out-of-fold scores of the 5 CV folds - and applied once to the test set. No test data is used to choose it.

**Test** (held-out 20%):
- threshold-free: `test_auc` (ROC-AUC), `test_ap` (average precision = AUPRC);
- at the Youden threshold: `test_f1` (F1 of the high-risk class - balance of precision and recall for the high-risk class), `test_f1_macro` (unweighted mean of the two per-class F1s), `test_balanced_accuracy` (mean of sensitivity and specificity), `test_sensitivity` (recall of the high-risk class), `test_specificity` (recall of the non high-risk class).

In [ ]:
from sklearn.base import clone
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve, f1_score,
                             balanced_accuracy_score, recall_score)
from joblib import Parallel, delayed

# SVC(probability=True) triggers an expensive internal 5-fold Platt-scaling calibration on
# every fit - far too slow at this sample size across many CV folds/targets. Use
# probability=False and score via decision_function instead (much faster, and sklearn's
# roc_auc scorer/our helper below both support it).
CLASSIFICATION_MODELS = {
    "logistic_regression": LogisticRegression(solver="liblinear", max_iter=1000, random_state=RANDOM_STATE),
    "svm_linear": LinearSVC(dual=False, max_iter=5000, random_state=RANDOM_STATE),
    "svm_rbf": SVC(kernel="rbf", random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    "lightgbm": LGBMClassifier(subsample_freq=1, random_state=RANDOM_STATE, n_jobs=1, verbosity=-1),
}

SELECTOR_SPACE = {"select__estimator__C": loguniform(1e-2, 10)}
CLASSIFICATION_SPACES = {
    "logistic_regression": {"model__C": loguniform(1e-3, 1e2), "model__penalty": ["l1", "l2"]},
    "svm_linear": {"model__C": loguniform(1e-4, 10)},
    "svm_rbf": {"model__C": loguniform(0.1, 30), "model__gamma": loguniform(1e-4, 1)},
    "random_forest": {
        "model__n_estimators": randint(200, 800),
        "model__max_depth": [None, 5, 8, 12, 20],
        "model__min_samples_leaf": randint(1, 60),
        "model__max_features": ["sqrt", 0.3, 0.5, 0.7],
    },
    "lightgbm": {
        "model__n_estimators": randint(100, 1000),
        "model__learning_rate": loguniform(5e-3, 0.2),
        "model__num_leaves": randint(4, 64),
        "model__min_child_samples": randint(10, 200),
        "model__subsample": uniform(0.5, 0.5),
        "model__colsample_bytree": uniform(0.5, 0.5),
        "model__reg_lambda": loguniform(1e-2, 100),
    },
}
CLASSIFICATION_SPACES = {k: {**v, **SELECTOR_SPACE} for k, v in CLASSIFICATION_SPACES.items()}

SPLITS = ["top20", "top10", "top5"]
CLASSIFICATION_TARGETS = [f"target_{t}_{s}" for t in TARGET_NAMES for s in SPLITS]
assert set(CLASSIFICATION_TARGETS) <= set(data.columns) and len(CLASSIFICATION_TARGETS) == 15


def get_scores(fitted_pipe, X_input):
    if hasattr(fitted_pipe, "predict_proba"):
        return fitted_pipe.predict_proba(X_input)[:, 1]
    return fitted_pipe.decision_function(X_input)


def _fold_scores(pipe, X, y, train_idx, val_idx, score_fn):
    fitted = clone(pipe).fit(X.iloc[train_idx], y.iloc[train_idx])
    return val_idx, score_fn(fitted, X.iloc[val_idx])


def oof_scores(pipe, X, y, cv, score_fn):
    # CV of the tuned configuration on the training split: out-of-fold scores for every
    # training row, plus each fold's row positions (to score the folds separately)
    folds = Parallel(n_jobs=-1)(delayed(_fold_scores)(pipe, X, y, tr, va, score_fn) for tr, va in cv.split(X, y))
    oof = np.empty((len(y),) + np.shape(folds[0][1])[1:])
    for val_idx, scores in folds:
        oof[val_idx] = scores
    return oof, [val_idx for val_idx, _ in folds]


def youden_threshold(y, scores):
    # cut-off maximizing sensitivity + specificity - 1
    fpr, tpr, thr = roc_curve(y, scores)
    return thr[np.argmax(tpr - fpr)]


clf_results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_skf = StratifiedKFold(n_splits=INNER_CV, shuffle=True, random_state=RANDOM_STATE)

for target_col in CLASSIFICATION_TARGETS:
    mask = data[target_col].notna()
    X_valid, y_valid = features_for(target_col)[mask], data.loc[mask, target_col].astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X_valid, y_valid, test_size=0.2, random_state=RANDOM_STATE, stratify=y_valid
    )

    for name, estimator in CLASSIFICATION_MODELS.items():
        t0 = time.time()
        pipe = make_pipeline(estimator, task="classification")
        best_pipe, inner_best, best_params = run_search(
            pipe, CLASSIFICATION_SPACES[name], X_train, y_train, inner_skf, "roc_auc"
        )
        oof, folds = oof_scores(best_pipe, X_train, y_train, skf, get_scores)
        cv_auc = [roc_auc_score(y_train.iloc[va], oof[va]) for va in folds]
        cv_ap = [average_precision_score(y_train.iloc[va], oof[va]) for va in folds]
        threshold = youden_threshold(y_train, oof)
        y_score = get_scores(best_pipe, X_test)
        y_pred = (y_score >= threshold).astype(int)

        clf_results.append({
            "target": target_col,
            "model": name,
            "n": mask.sum(),
            "pos_rate": y_valid.mean(),
            "inner_cv_best": inner_best,
            "cv_auc_mean": np.mean(cv_auc),
            "cv_auc_std": np.std(cv_auc),
            "cv_ap_mean": np.mean(cv_ap),
            "cv_ap_std": np.std(cv_ap),
            "threshold": threshold,
            "test_auc": roc_auc_score(y_test, y_score),
            "test_ap": average_precision_score(y_test, y_score),
            "test_f1": f1_score(y_test, y_pred),
            "test_f1_macro": f1_score(y_test, y_pred, average="macro"),
            "test_balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "test_sensitivity": recall_score(y_test, y_pred),
            "test_specificity": recall_score(y_test, y_pred, pos_label=0),
            "best_params": str(best_params),
        })
        r = clf_results[-1]
        print(f"{target_col:32s} {name:20s} cv_auc={r['cv_auc_mean']:.3f}  test_auc={r['test_auc']:.3f}"
              f"  test_ap={r['test_ap']:.3f}  sens={r['test_sensitivity']:.3f}  spec={r['test_specificity']:.3f}"
              f"  ({time.time() - t0:.0f}s)", flush=True)

clf_results_df = pd.DataFrame(clf_results)

In [ ]:
clf_results_df.sort_values(["target", "test_auc"], ascending=[True, False])

In [ ]:
import os
SUPPLE_TABLE_DIR = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Github2/supple_tables"
os.makedirs(SUPPLE_TABLE_DIR, exist_ok=True)
reg_results_df.to_csv(f"{NOTEBOOK_DIR}/step1_regression_results.csv", index=False)
clf_results_df.to_csv(f"{NOTEBOOK_DIR}/step1_classification_results.csv", index=False)
reg_results_df.to_csv(f"{SUPPLE_TABLE_DIR}/regression_results.csv", index=False)
clf_results_df.to_csv(f"{SUPPLE_TABLE_DIR}/classification_results.csv", index=False)
print("saved regression and classification result tables")

import shutil
shutil.rmtree(CACHE_DIR, ignore_errors=True)

## Feature importance (classification track)

Which features drive the high-risk classification? Feature importance is computed for **logistic regression** on the three binary labels - **top 20%**, **top 10%** and **top 5%** vs the rest (High-risk vs Non high-risk) - for each of the 5 brain atrophy targets (15 labels). The tuning loop above keeps only metrics, so each model is **refit** here from its saved `best_params` on the exact same train/test split (same `random_state` + stratification), and the refit test AUC is checked against the stored value. Two complementary measures:

1. **Permutation importance (ΔAUC)** - on the held-out test split. Each raw input column is shuffled (10 repeats) and the drop in test AUC is recorded. Columns are permuted *before* the pipeline, so a categorical feature's one-hot dummies move together and importance is reported per original feature (e.g. `apoe_geno_group`). Features removed by the L1 selector get ΔAUC = 0.
2. **Effect size - standardized odds ratios** from the same tuned logistic regressions: OR per 1 SD for numeric features (they are standardized in the pipeline), OR of each category vs. the rest for one-hot dummies. Gives the **direction** that permutation importance lacks. Dropped features show coef = 0 / OR = 1.

In [ ]:
import ast
import os
import tempfile
from joblib import Memory
from sklearn.inspection import permutation_importance

# The last cell deleted the pipeline cache dir - point make_pipeline at a fresh one
CACHE_DIR = tempfile.mkdtemp(prefix="step1_pipe_cache_")
memory = Memory(CACHE_DIR, verbose=0)


def params_from_str(s):
    # inverse of the key renaming in run_search
    return {("select__estimator__C" if k == "selector_C" else f"model__{k}"): v
            for k, v in ast.literal_eval(s).items()}


def refit(target_col, model_name):
    # same split as the classification loop
    row = clf_results_df[(clf_results_df["target"] == target_col) & (clf_results_df["model"] == model_name)].iloc[0]
    mask = data[target_col].notna()
    X_valid, y_valid = features_for(target_col)[mask], data.loc[mask, target_col].astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X_valid, y_valid, test_size=0.2, random_state=RANDOM_STATE, stratify=y_valid
    )
    pipe = make_pipeline(CLASSIFICATION_MODELS[model_name], task="classification")
    pipe.set_params(**params_from_str(row["best_params"])).fit(X_train, y_train)
    refit_auc = roc_auc_score(y_test, get_scores(pipe, X_test))
    assert abs(refit_auc - row["test_auc"]) < 1e-3, (target_col, model_name, refit_auc, row["test_auc"])
    return pipe, X_test, y_test


def expanded_feature_names(pipe):
    # sklearn 0.24: no get_feature_names_out on ColumnTransformer
    onehot = pipe.named_steps["preprocess"].named_transformers_["cat"].named_steps["onehot"]
    names = np.array(NUMERIC_COLS + list(onehot.get_feature_names(CATEGORICAL_COLS)))
    return names[pipe.named_steps["select"].get_support()]

In [ ]:
FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS
FI_MODEL = "logistic_regression"
FI_SPLITS = ["top20", "top10", "top5"]
FI_TARGETS = [f"target_{t}_{s}" for s in FI_SPLITS for t in TARGET_NAMES]

perm_rows = []
for target_col in FI_TARGETS:
    t0 = time.time()
    pipe, X_test, y_test = refit(target_col, FI_MODEL)
    pi = permutation_importance(pipe, X_test, y_test, scoring="roc_auc", n_repeats=10,
                                random_state=RANDOM_STATE, n_jobs=-1)
    for feat, m, s in zip(FEATURE_COLS, pi.importances_mean, pi.importances_std):
        perm_rows.append({"target": target_col, "model": FI_MODEL, "feature": feat,
                          "imp_mean": m, "imp_std": s})
    print(f"{target_col:42s} {FI_MODEL:20s} ({time.time() - t0:.0f}s)", flush=True)

perm_df = pd.DataFrame(perm_rows)
perm_imp = perm_df.pivot(index="feature", columns="target", values="imp_mean")[FI_TARGETS]

ranks = perm_imp.rank(ascending=False)
perm_summary = pd.DataFrame({
    "mean_dAUC": perm_imp.mean(axis=1),
    "sd_dAUC": perm_imp.std(axis=1),
    "max_dAUC": perm_imp.max(axis=1),
    "mean_rank": ranks.mean(axis=1),
    "n_top3": (ranks <= 3).sum(axis=1),
}).sort_values("mean_dAUC", ascending=False)
perm_summary.round(4)

In [ ]:
# Top 5 features per label (logistic regression ΔAUC)
pd.DataFrame({t.replace("target_", ""): [f"{f} ({v:.3f})" for f, v in perm_imp[t].nlargest(5).items()]
              for t in FI_TARGETS},
             index=pd.RangeIndex(1, 6, name="rank")).T

In [ ]:
coef_rows = []
for target_col in FI_TARGETS:
    pipe, _, _ = refit(target_col, FI_MODEL)
    for feat, c in zip(expanded_feature_names(pipe), pipe.named_steps["model"].coef_[0]):
        coef_rows.append({"target": target_col, "feature": feat, "coef": c, "odds_ratio": np.exp(c)})

coef_df = pd.DataFrame(coef_rows)
# features dropped by the L1 selector (or zeroed by an L1 model) -> coef 0
logit_coef = coef_df.pivot(index="feature", columns="target", values="coef").reindex(columns=FI_TARGETS).fillna(0)
logit_summary = pd.DataFrame({
    "mean_coef": logit_coef.mean(axis=1),
    "mean_OR": np.exp(logit_coef.mean(axis=1)),
    "mean_abs_coef": logit_coef.abs().mean(axis=1),
    "n_selected": (logit_coef != 0).sum(axis=1),
}).sort_values("mean_abs_coef", ascending=False)
logit_summary.round(3)

In [ ]:
# standardized odds ratio per label (1 = dropped by the L1 selector)
logit_or = np.exp(logit_coef.loc[logit_summary.index])
logit_or.columns = [c.replace("target_", "") for c in logit_or.columns]
logit_or.round(3)

In [ ]:
# Age, hypertension and sex per label: permutation ΔAUC (method 1) and odds ratio (method 2)
# OR: age per 1 SD; hypertension yes vs no and female vs male as the contrast of the two one-hot
# coefficients (both dummies are in the model, so a single dummy's OR is only "vs the rest")
coef = logit_coef.reindex(["T_AGE", "T_HTN_1.0", "T_HTN_2.0", "T_SEX_1", "T_SEX_2"]).fillna(0)
key_rows = []
for target_col in FI_TARGETS:
    split = target_col.rsplit("_", 1)[1]
    name = target_col[len("target_"):-len(split) - 1]
    c = coef[target_col]
    key_rows.append({
        "split": split, "target": name,
        "dAUC_age": perm_imp.loc["T_AGE", target_col],
        "dAUC_htn": perm_imp.loc["T_HTN", target_col],
        "dAUC_sex": perm_imp.loc["T_SEX", target_col],
        "OR_age_per_SD": np.exp(c["T_AGE"]),
        "OR_htn_yes_vs_no": np.exp(c["T_HTN_2.0"] - c["T_HTN_1.0"]),
        "OR_female_vs_male": np.exp(c["T_SEX_2"] - c["T_SEX_1"]),
    })
key_effects = pd.DataFrame(key_rows).set_index(["split", "target"])

# per split: mean (range) over the 5 targets; ORs averaged on the log scale
key_summary = pd.DataFrame({
    col: key_effects[col].groupby(level="split").apply(
        lambda s: f"{np.exp(np.log(s).mean()) if col.startswith('OR') else s.mean():.3f} "
                  f"({s.min():.3f}-{s.max():.3f})")
    for col in key_effects.columns
}).loc[FI_SPLITS]
display(key_effects.round(3))
key_summary

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

FI_PLOT_TARGET = "composite"
FI_SPLIT_LABELS = {"top20": "Top 20% vs rest", "top10": "Top 10% vs rest", "top5": "Top 5% vs rest"}

fig, axes = plt.subplots(1, len(FI_SPLITS), figsize=(7 * len(FI_SPLITS), 9))
for ax, split in zip(axes, FI_SPLITS):
    imp = perm_df[perm_df["target"] == f"target_{FI_PLOT_TARGET}_{split}"].set_index("feature")
    imp = imp.sort_values("imp_mean")
    ax.barh(imp.index, imp["imp_mean"], xerr=imp["imp_std"], color="#4C72B0", ecolor="0.6", capsize=2)
    ax.set_xlabel("Permutation importance: mean drop in test AUC (± SD over 10 repeats)")
    ax.set_title(f"Logistic regression - Composite, {split}\n({FI_SPLIT_LABELS[split]})")
plt.tight_layout()
plt.show()

In [ ]:
perm_df.to_csv(f"{NOTEBOOK_DIR}/step1_clf_permutation_importance.csv", index=False)
coef_df.to_csv(f"{NOTEBOOK_DIR}/step1_clf_logit_coefficients.csv", index=False)
print("saved permutation importance and logistic coefficient tables")
shutil.rmtree(CACHE_DIR, ignore_errors=True)

### Interpretation

*(Results omitted from the public repository under the Healthcare AI Challenge 2026 synthetic-data use agreement.)*

## ROC curves by target (top 20%, 10% and 5% labels)

For one split at a time, each classifier is refit from its saved `best_params` (same train/test split, via `refit` above) for all 5 targets, and its held-out test ROC curve is drawn. One panel per model, five lines per panel (one per target); each target keeps the same color in every panel. Figures are saved as PNG to `Figures/20261021_Github2`; the combined 5-model panels are also saved to `Github2/supple_figures` (`top20_auc.png`, `top10_auc.png`, `top5_auc.png`).

In [ ]:
import os
import tempfile
from joblib import Memory
from sklearn.metrics import roc_curve

# The feature-importance section deleted its pipeline cache dir - point make_pipeline at a fresh one
CACHE_DIR = tempfile.mkdtemp(prefix="step1_pipe_cache_")
memory = Memory(CACHE_DIR, verbose=0)

FIG_DIR = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Figures/20261021_Github2"
os.makedirs(FIG_DIR, exist_ok=True)
SUPPLE_FIG_DIR = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Github2/supple_figures"
os.makedirs(SUPPLE_FIG_DIR, exist_ok=True)

TARGET_LABELS = {
    "composite": "Composite",
    "wholebrain": "Whole brain",
    "hippo": "Hippocampus",
    "ventricles": "Ventricles",
    "amygdala": "Subcortical amygdala",
}
# fixed per-target colors, shared by every panel
TARGET_COLORS = dict(zip(TARGET_LABELS, sns.color_palette("tab10", len(TARGET_LABELS))))
MODEL_LABELS = {
    "logistic_regression": "Logistic Regression",
    "svm_linear": "SVM-Linear",
    "svm_rbf": "SVM-RBF",
    "random_forest": "Random Forest",
    "lightgbm": "LightGBM",
}
# every text element (titles, axis labels, ticks, legend) at the same size
FONT_SIZE = 16
PANEL_IN = 8  # panel width/height in inches - large enough that 16 pt legends leave the curves visible
FONT_RC = {k: FONT_SIZE for k in ["font.size", "axes.titlesize", "axes.labelsize",
                                  "xtick.labelsize", "ytick.labelsize", "legend.fontsize"]}


def roc_curves_for_split(split):
    # {model: {target: (fpr, tpr, auc)}} on the held-out test split
    curves = {m: {} for m in MODEL_LABELS}
    for t in TARGET_LABELS:
        for m in MODEL_LABELS:
            pipe, X_test, y_test = refit(f"target_{t}_{split}", m)
            scores = get_scores(pipe, X_test)
            fpr, tpr, _ = roc_curve(y_test, scores)
            curves[m][t] = (fpr, tpr, roc_auc_score(y_test, scores))
    return curves


def draw_roc(ax, model_curves, title):
    for t, (fpr, tpr, auc) in model_curves.items():
        ax.plot(fpr, tpr, color=TARGET_COLORS[t], lw=1.8, label=f"{TARGET_LABELS[t]} (AUC = {auc:.3f})")
    ax.plot([0, 1], [0, 1], ls="--", color="0.6", lw=1)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.01)
    ax.set_aspect("equal")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(title)
    ax.legend(loc="lower right")


def plot_split_rocs(split, split_label, supple_name=None):
    curves = roc_curves_for_split(split)
    with plt.rc_context(FONT_RC):
        _save_split_figures(split, split_label, curves, supple_name)
    return curves


def _save_split_figures(split, split_label, curves, supple_name=None):
    # one PNG per model
    for m, model_curves in curves.items():
        fig, ax = plt.subplots(figsize=(PANEL_IN, PANEL_IN))
        draw_roc(ax, model_curves, f"{MODEL_LABELS[m]} - {split_label}")
        fig.tight_layout()
        fig.savefig(f"{FIG_DIR}/step1_roc_{split}_{m}.png", dpi=300, bbox_inches="tight")
        plt.close(fig)
    # combined panel of all 5 models
    fig, axes = plt.subplots(1, len(curves), figsize=(PANEL_IN * len(curves), PANEL_IN))
    for ax, (m, model_curves) in zip(axes, curves.items()):
        draw_roc(ax, model_curves, MODEL_LABELS[m])
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/step1_roc_{split}_all_models.png", dpi=300, bbox_inches="tight")
    if supple_name:
        fig.savefig(f"{SUPPLE_FIG_DIR}/{supple_name}", dpi=300, bbox_inches="tight")
    plt.show()

### Top 20%

Top 20% vs the rest (positive rate 0.20).

In [ ]:
roc_top20 = plot_split_rocs("top20", "Top 20% vs rest", supple_name="top20_auc.png")

### Top 10%

Top 10% vs the rest (positive rate 0.10).

In [ ]:
roc_top10 = plot_split_rocs("top10", "Top 10% vs rest", supple_name="top10_auc.png")

### Top 5%

Top 5% vs the rest (positive rate 0.05).

In [ ]:
roc_top5 = plot_split_rocs("top5", "Top 5% vs rest", supple_name="top5_auc.png")
shutil.rmtree(CACHE_DIR, ignore_errors=True)
print(f"saved ROC figures to {FIG_DIR} and {SUPPLE_FIG_DIR}")

## Multiclass track (ordered risk classes)

Labels: `risk5` (quintiles, 5 = top 20% ... 1 = bottom 20%) and `risk4` (quartiles, 4 = top 25% ... 1 = bottom 25%) for each of the 5 targets (10 labels, full N = 10,000, equal-size classes). Same stratified 80/20 split, pipeline and L1-logistic feature selector as the binary track.

Two modeling approaches:
- **Standard (nominal) classification** - the 5 binary-track model families in their multiclass form: multinomial logistic regression (`saga`, L1/L2), linear SVM (one-vs-rest), RBF SVM, random forest, LightGBM. The class order is ignored: predicting risk1 for a risk5 participant counts the same as predicting risk4.
- **Ordinal classification** - **proportional-odds (ordinal) logistic regression**: one linear risk score x·β and K-1 ordered cut points, P(y ≤ k | x) = σ(θ_k − x·β). It uses the class order and gives one odds ratio per feature (odds of being in a higher risk class), shared by all cut points. Implemented below as a small scikit-learn estimator with a ridge penalty α (tuned), so it runs through the same pipeline / search / CV code as the other models.

**Metrics** - identical for both approaches so they can be compared directly:
- Hyperparameter search and validation: **quadratic weighted kappa (QWK)** - agreement between predicted and true class that penalizes an error by the squared class distance (risk5 → risk1 costs 16x more than risk5 → risk4); 0 = chance, 1 = perfect. Validation also reports the **macro one-vs-rest ROC-AUC** (mean over classes of the AUC of "this class vs the rest"; threshold-free, the multiclass counterpart of the binary ROC-AUC).
- Test: `test_qwk` (primary), `test_mae` (mean absolute class error, in classes), `test_macro_auc`, `test_f1_macro`, `test_balanced_accuracy` (mean per-class recall; chance = 1/K), and `test_top_sensitivity` / `test_top_specificity` - recall of the highest-risk class and of everyone else, from the predicted classes (links to the top 20% / top 25% screening question).

Predicted class = the model's `predict` (argmax of the class probabilities for the ordinal model). For the SVMs, which have no probabilities, the per-class decision values are used for the AUC; the per-class AUC only needs a ranking, not calibrated probabilities.

In [ ]:
from scipy import sparse
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, ClassifierMixin


class OrdinalLogisticRegression(BaseEstimator, ClassifierMixin):
    # Proportional-odds (cumulative logit) model: P(y <= k | x) = sigmoid(theta_k - x @ beta), so a
    # larger x @ beta shifts probability to higher (more atrophy) classes, with one beta shared by all
    # cut points. Ridge penalty alpha * ||beta||^2 on the summed negative log-likelihood (thresholds
    # unpenalized) - it also absorbs the collinearity of full one-hot dummies with the thresholds.
    # Thresholds are parameterized as theta_1 plus cumulative exp-increments so they stay ordered.
    def __init__(self, alpha=1.0, max_iter=2000):
        self.alpha = alpha
        self.max_iter = max_iter

    @staticmethod
    def _as_dense(X):
        return X.toarray() if sparse.issparse(X) else np.asarray(X, dtype=float)

    @staticmethod
    def _thresholds(a):
        return np.cumsum(np.concatenate([a[:1], np.exp(a[1:])]))

    @staticmethod
    def _cum_probs(eta, theta):
        # n x (K+1): column k = P(y <= k-th class - 1), padded with 0 / 1 at the ends
        # (imported here, not at module level: joblib workers cannot unpickle a notebook-level
        # reference to the scipy ufunc)
        from scipy.special import expit
        F = expit(theta[None, :] - eta[:, None])
        return np.hstack([np.zeros((len(eta), 1)), F, np.ones((len(eta), 1))])

    def _loss_grad(self, params, X, yk, K):
        p = X.shape[1]
        beta, a = params[:p], params[p:]
        F = self._cum_probs(X @ beta, self._thresholds(a))
        rows = np.arange(len(yk))
        upper, lower = F[rows, yk + 1], F[rows, yk]
        prob = np.clip(upper - lower, 1e-12, None)
        f_up, f_lo = upper * (1 - upper), lower * (1 - lower)  # logistic density at the two cut points
        loss = -np.log(prob).sum() + self.alpha * beta @ beta

        g_beta = X.T @ ((f_up - f_lo) / prob) + 2 * self.alpha * beta
        g_theta = np.zeros(K - 1)
        np.add.at(g_theta, yk[yk < K - 1], -(f_up / prob)[yk < K - 1])
        np.add.at(g_theta, yk[yk > 0] - 1, (f_lo / prob)[yk > 0])
        # chain rule through theta = cumsum([a_0, exp(a_1), ...])
        tail = np.cumsum(g_theta[::-1])[::-1]
        g_a = np.concatenate([tail[:1], tail[1:] * np.exp(a[1:])])
        return loss, np.concatenate([g_beta, g_a])

    def fit(self, X, y):
        X = self._as_dense(X)
        self.classes_, yk = np.unique(y, return_inverse=True)
        K = len(self.classes_)
        # start at beta = 0 with thresholds at the logits of the cumulative class proportions
        cum = np.cumsum(np.bincount(yk, minlength=K))[:-1] / len(yk)
        theta0 = np.log(cum / (1 - cum))
        x0 = np.concatenate([np.zeros(X.shape[1]), theta0[:1], np.log(np.diff(theta0))])
        res = minimize(self._loss_grad, x0, args=(X, yk, K), jac=True, method="L-BFGS-B",
                       options={"maxiter": self.max_iter})
        self.coef_ = res.x[:X.shape[1]]
        self.thresholds_ = self._thresholds(res.x[X.shape[1]:])
        self.converged_ = res.success
        return self

    def predict_proba(self, X):
        return np.diff(self._cum_probs(self._as_dense(X) @ self.coef_, self.thresholds_), axis=1)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

### Sanity check of the ordinal estimator

1. With K = 2 classes the proportional-odds model **is** binary logistic regression (β equal, cut point = −intercept). Fit both, unpenalized, on `target_composite_top20` with a few features and compare against `statsmodels` `Logit`.
2. With K = 5 (`target_composite_risk5`) the analytic gradient of the penalized log-likelihood is compared with finite differences.

(`statsmodels.miscmodels.ordinal_model.OrderedModel` would be the natural reference, but it needs statsmodels ≥ 0.13; this environment has 0.12.2.)

In [ ]:
import statsmodels.api as sm
from scipy.optimize import check_grad

chk_cols = ["T_AGE", "T_SBP", "T_BMI", "T_HDL"]
X_chk = data[chk_cols].fillna(data[chk_cols].median())
X_chk = ((X_chk - X_chk.mean()) / X_chk.std()).assign(female=(data["T_SEX"] == 2).astype(float))

# (1) K = 2 vs statsmodels Logit
y2 = data["target_composite_top20"].values
olr2 = OrdinalLogisticRegression(alpha=0).fit(X_chk.values, y2)
logit = sm.Logit(y2, sm.add_constant(X_chk)).fit(disp=0)
k2_check = pd.DataFrame({
    "ordinal_K2": np.r_[olr2.coef_, olr2.thresholds_],
    "statsmodels_logit": np.r_[logit.params.values[1:], -logit.params.values[0]],
}, index=list(X_chk.columns) + ["cut point (= -intercept)"])
display(k2_check.round(5))
assert np.allclose(k2_check["ordinal_K2"], k2_check["statsmodels_logit"], atol=1e-3)

# (2) K = 5 gradient check
y5 = data["target_composite_risk5"].values
_, yk5 = np.unique(y5, return_inverse=True)
olr5 = OrdinalLogisticRegression(alpha=1.0)
w0 = np.random.default_rng(RANDOM_STATE).normal(scale=0.1, size=X_chk.shape[1] + 4)
grad_err = check_grad(lambda w: olr5._loss_grad(w, X_chk.values, yk5, 5)[0],
                      lambda w: olr5._loss_grad(w, X_chk.values, yk5, 5)[1], w0)
rel_err = grad_err / np.linalg.norm(olr5._loss_grad(w0, X_chk.values, yk5, 5)[1])
print(f"relative gradient error (K = 5): {rel_err:.2e}")
assert rel_err < 1e-5

In [ ]:
import tempfile
from joblib import Memory
from sklearn.metrics import make_scorer, cohen_kappa_score, confusion_matrix

# The ROC section deleted its pipeline cache dir - point make_pipeline at a fresh one
CACHE_DIR = tempfile.mkdtemp(prefix="step1_pipe_cache_")
memory = Memory(CACHE_DIR, verbose=0)

MULTICLASS_MODELS = {
    "logistic_regression": LogisticRegression(solver="saga", multi_class="multinomial", max_iter=2000,
                                              random_state=RANDOM_STATE),
    "svm_linear": CLASSIFICATION_MODELS["svm_linear"],
    "svm_rbf": CLASSIFICATION_MODELS["svm_rbf"],
    "random_forest": CLASSIFICATION_MODELS["random_forest"],
    "lightgbm": CLASSIFICATION_MODELS["lightgbm"],
    "ordinal_logistic": OrdinalLogisticRegression(),
}
# same search spaces as the binary track; the ordinal model tunes its ridge penalty
MULTICLASS_SPACES = {
    **CLASSIFICATION_SPACES,
    "ordinal_logistic": {"model__alpha": loguniform(1e-3, 1e3), **SELECTOR_SPACE},
}
QWK_SCORER = make_scorer(cohen_kappa_score, weights="quadratic")

RISK_SPLITS = ["risk5", "risk4"]
MULTICLASS_TARGETS = [f"target_{t}_{s}" for t in TARGET_NAMES for s in RISK_SPLITS]
assert set(MULTICLASS_TARGETS) <= set(data.columns) and len(MULTICLASS_TARGETS) == 10


def class_scores(fitted_pipe, X_input):
    # n x K per-class scores: probabilities, or one-vs-rest decision values for the SVMs
    if hasattr(fitted_pipe, "predict_proba"):
        return fitted_pipe.predict_proba(X_input)
    return fitted_pipe.decision_function(X_input)


def pred_and_scores(fitted_pipe, X_input):
    # column 0 = predicted class, columns 1.. = per-class scores
    return np.column_stack([fitted_pipe.predict(X_input), class_scores(fitted_pipe, X_input)])


def macro_ovr_auc(y, scores, classes):
    return np.mean([roc_auc_score(y == c, scores[:, j]) for j, c in enumerate(classes)])


def multiclass_metrics(y, y_pred, scores, classes):
    y, y_pred = np.asarray(y), np.asarray(y_pred).astype(int)
    top = classes[-1]
    return {
        "qwk": cohen_kappa_score(y, y_pred, weights="quadratic"),
        "mae": np.mean(np.abs(y - y_pred)),
        "macro_auc": macro_ovr_auc(y, scores, classes),
        "f1_macro": f1_score(y, y_pred, average="macro"),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "top_sensitivity": recall_score(y == top, y_pred == top),
        "top_specificity": recall_score(y != top, y_pred != top),
    }

In [ ]:
mc_results = []
for target_col in MULTICLASS_TARGETS:
    X, y = features_for(target_col), data[target_col].astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    classes = np.sort(y.unique())

    for name, estimator in MULTICLASS_MODELS.items():
        t0 = time.time()
        pipe = make_pipeline(estimator, task="classification")
        best_pipe, inner_best, best_params = run_search(
            pipe, MULTICLASS_SPACES[name], X_train, y_train, inner_skf, QWK_SCORER
        )
        oof, folds = oof_scores(best_pipe, X_train, y_train, skf, pred_and_scores)
        cv_qwk = [cohen_kappa_score(y_train.iloc[va], oof[va, 0].astype(int), weights="quadratic") for va in folds]
        cv_mauc = [macro_ovr_auc(y_train.iloc[va].values, oof[va, 1:], classes) for va in folds]
        test = multiclass_metrics(y_test, best_pipe.predict(X_test), class_scores(best_pipe, X_test), classes)

        mc_results.append({
            "target": target_col,
            "model": name,
            "approach": "ordinal" if name == "ordinal_logistic" else "nominal",
            "n_classes": len(classes),
            "inner_cv_best": inner_best,
            "cv_qwk_mean": np.mean(cv_qwk),
            "cv_qwk_std": np.std(cv_qwk),
            "cv_macro_auc_mean": np.mean(cv_mauc),
            "cv_macro_auc_std": np.std(cv_mauc),
            **{f"test_{k}": v for k, v in test.items()},
            "best_params": str(best_params),
        })
        r = mc_results[-1]
        print(f"{target_col:32s} {name:20s} cv_qwk={r['cv_qwk_mean']:.3f}  test_qwk={r['test_qwk']:.3f}"
              f"  test_mae={r['test_mae']:.3f}  test_macro_auc={r['test_macro_auc']:.3f}"
              f"  ({time.time() - t0:.0f}s)", flush=True)

mc_results_df = pd.DataFrame(mc_results)
mc_results_df.to_csv(f"{NOTEBOOK_DIR}/step1_multiclass_results.csv", index=False)
mc_results_df.to_csv(f"{SUPPLE_TABLE_DIR}/multiclass_results.csv", index=False)
print("saved", mc_results_df.shape)

In [ ]:
# test QWK per label (rows) and model (columns)
mc_view = mc_results_df.assign(label=mc_results_df["target"].str.replace("target_", ""))
mc_view.pivot(index="label", columns="model", values="test_qwk") \
    .reindex([t.replace("target_", "") for t in MULTICLASS_TARGETS])[list(MULTICLASS_MODELS)].round(3)

In [ ]:
# all test metrics, averaged over the 5 targets, per risk scheme and model
metric_cols = ["cv_qwk_mean", "cv_macro_auc_mean"] + [c for c in mc_results_df.columns if c.startswith("test_")]
mc_view.assign(scheme=mc_view["label"].str.rsplit("_", n=1).str[1]) \
    .groupby(["scheme", "model"])[metric_cols].mean().reindex(list(MULTICLASS_MODELS), level="model").round(3)

### Confusion matrices (composite target)

Row-normalized test confusion matrices (each row = one true class, sums to 1) for the composite target: the best nominal model (highest validation QWK, chosen without the test set) next to the ordinal logistic regression. Mass close to the diagonal = small class errors. Saved to `Github2/supple_figures` (`risk5_confusion.png`, `risk4_confusion.png`).

In [ ]:
def refit_multiclass(target_col, model_name):
    # same split as the multiclass loop
    row = mc_results_df[(mc_results_df["target"] == target_col) & (mc_results_df["model"] == model_name)].iloc[0]
    X, y = features_for(target_col), data[target_col].astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    pipe = make_pipeline(MULTICLASS_MODELS[model_name], task="classification")
    pipe.set_params(**params_from_str(row["best_params"])).fit(X_train, y_train)
    refit_qwk = cohen_kappa_score(y_test, pipe.predict(X_test), weights="quadratic")
    assert abs(refit_qwk - row["test_qwk"]) < 1e-3, (target_col, model_name, refit_qwk, row["test_qwk"])
    return pipe, X_test, y_test


MC_MODEL_LABELS = {**MODEL_LABELS, "ordinal_logistic": "Ordinal LR"}
MC_PLOT_TARGET = "composite"

for scheme in RISK_SPLITS:
    target_col = f"target_{MC_PLOT_TARGET}_{scheme}"
    sub = mc_results_df[(mc_results_df["target"] == target_col) & (mc_results_df["approach"] == "nominal")]
    best_nominal = sub.loc[sub["cv_qwk_mean"].idxmax(), "model"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, model_name in zip(axes, [best_nominal, "ordinal_logistic"]):
        pipe, X_test, y_test = refit_multiclass(target_col, model_name)
        classes = np.sort(y_test.unique())
        cm = confusion_matrix(y_test, pipe.predict(X_test), labels=classes, normalize="true")
        ticks = [f"risk{c}" for c in classes]
        sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False,
                    xticklabels=ticks, yticklabels=ticks, ax=ax)
        qwk = cohen_kappa_score(y_test, pipe.predict(X_test), weights="quadratic")
        ax.set_xlabel("Predicted class")
        ax.set_ylabel("True class")
        ax.set_title(f"{MC_MODEL_LABELS[model_name]} - {TARGET_LABELS[MC_PLOT_TARGET]}, {scheme} (QWK = {qwk:.3f})")
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/step1_confusion_{scheme}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{SUPPLE_FIG_DIR}/{scheme}_confusion.png", dpi=300, bbox_inches="tight")
    plt.show()

### Ordinal logistic regression: odds ratios

The proportional-odds model has **one** coefficient per feature, shared by all cut points: exp(β) is the odds ratio of being in a **higher** risk class (any cut point) per 1 SD of a numeric feature, or for a category vs the rest. Features dropped by the L1 selector show OR = 1. Age, hypertension and sex are summarized with the same contrasts as the binary feature-importance section.

In [ ]:
ord_rows = []
for target_col in MULTICLASS_TARGETS:
    pipe, _, _ = refit_multiclass(target_col, "ordinal_logistic")
    for feat, cf in zip(expanded_feature_names(pipe), pipe.named_steps["model"].coef_):
        ord_rows.append({"target": target_col, "feature": feat, "coef": cf, "odds_ratio": np.exp(cf)})

ord_coef_df = pd.DataFrame(ord_rows)
ord_coef = ord_coef_df.pivot(index="feature", columns="target", values="coef") \
    .reindex(columns=MULTICLASS_TARGETS).fillna(0)
ord_order = ord_coef.abs().mean(axis=1).sort_values(ascending=False).index
ord_or = np.exp(ord_coef.loc[ord_order])
ord_or.columns = [c.replace("target_", "") for c in ord_or.columns]
ord_or.round(3)

In [ ]:
oc = ord_coef.reindex(["T_AGE", "T_HTN_1.0", "T_HTN_2.0", "T_SEX_1", "T_SEX_2"]).fillna(0)
ord_key = pd.DataFrame([{
    "split": target_col.rsplit("_", 1)[1],
    "target": target_col[len("target_"):].rsplit("_", 1)[0],
    "OR_age_per_SD": np.exp(oc.loc["T_AGE", target_col]),
    "OR_htn_yes_vs_no": np.exp(oc.loc["T_HTN_2.0", target_col] - oc.loc["T_HTN_1.0", target_col]),
    "OR_female_vs_male": np.exp(oc.loc["T_SEX_2", target_col] - oc.loc["T_SEX_1", target_col]),
} for target_col in MULTICLASS_TARGETS]).set_index(["split", "target"])

ord_coef_df.to_csv(f"{NOTEBOOK_DIR}/step1_ordinal_logit_coefficients.csv", index=False)
shutil.rmtree(CACHE_DIR, ignore_errors=True)
ord_key.round(3)

### Interpretation (multiclass)

*(Results omitted from the public repository under the Healthcare AI Challenge 2026 synthetic-data use agreement.)*